# 2 — REDSEA pixel-level spillover correction

Corrects lateral marker spillover across shared cell boundaries. For each donor it
rasterizes the QuPath GeoJSON to an int32 mask, sums each qptiff channel per cell and
per 1-px boundary band, builds an 8-connected contact graph, and applies
`corrected = clip(data - alpha*(F@edge))` (subtract-only, alpha=1, 1-px band).

Every donor is independent, so set `[compute] n_jobs` > 1 (or use the SLURM array
`scripts/slurm/02_redsea_array.sh`). The optional CuPy backend (`use_gpu`) accelerates
the per-channel `bincount` and the sparse `F@edge` — use it with `n_jobs=1`.

In [ ]:
# Resolve the pipeline configuration (paths + params from ../config.ini).
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # make `phenocycler` importable from notebooks/
from phenocycler import load_config
cfg = load_config(pathlib.Path.cwd().parent / 'config.ini')
print('data_dir     :', cfg.data_dir)
print('images_dir   :', cfg.images_dir)
print('cells_csv    :', cfg.cells_csv)
print('n_jobs       :', cfg.n_jobs, '| use_gpu:', cfg.use_gpu)
cfg.discover_donors()  # donors found under data/cells/donor_id=* (empty until Step 1)

In [ ]:
from phenocycler.redsea import run_redsea, RedseaParams
params = RedseaParams.from_config(cfg)
print('comp_mode=%d alpha=%.2f edge_radius=%d gap_bridge=%d gpu=%s'
      % (params.comp_mode, params.alpha, params.edge_radius, params.gap_bridge, params.use_gpu))

### Smoke test one donor first (recommended)

In [ ]:
donors = cfg.discover_donors()
smoke = donors[:1]  # smallest/first donor
run_redsea(cfg, smoke, params, n_jobs=1)

### Then the full cohort

In [ ]:
run_redsea(cfg, cfg.discover_donors(), params, n_jobs=cfg.n_jobs)

### Validation figure (optional)

With `--save-intermediates` written during the run, sweep alpha and confirm impossible
cross-compartment co-positives collapse while single-lineage cells are preserved.
See `redsea_full_validate` in the source pipeline; the ported `compensate()` is reused.